# Course: Data Science Course
# Author: Sandro Camargo <sandrocamargo@unipampa.edu.br>
# Class 05 - Classification


To open this code in your Google Colab environment, [click here](https://colab.research.google.com/github/Sandrocamargo/data-science/blob/master/cd05_classificacao_titanic.ipynb).

# 📦 PACOTES


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report

# Faça o download do arquivo titanic.zip de https://www.kaggle.com/competitions/titanic/data

Após o download, faça o upload no seu ambiente do google colab.

In [ ]:
!unzip -o titanic.zip

# 📥 CARREGAR DADOS


In [ ]:
# coloque o caminho do arquivo train.csv do Kaggle
df = pd.read_csv("train.csv")
df.head()

# 🎯 VARIÁVEL TARGET


In [ ]:
y = df["Survived"]
X = df.drop(columns=["Survived", "Name", "Ticket", "Cabin"])

# 🔧 FEATURES


In [ ]:
num_features = ["Age", "Fare", "SibSp", "Parch"]
cat_features = ["Sex", "Embarked", "Pclass"]

# 🔄 PIPELINE DE PRÉ-PROCESSAMENTO


In [ ]:
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_features),
    ("cat", categorical_pipeline, cat_features)
])

# ✂️ SPLIT


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 🤖 MODELOS


In [ ]:
modelos = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=500, random_state=42)
}

# 🚀 TREINAMENTO E AVALIAÇÃO


In [ ]:
resultados = {}

for nome, modelo in modelos.items():

    pipeline = Pipeline([
        ("preprocessamento", preprocessor),
        ("modelo", modelo)
    ])

    # treino
    pipeline.fit(X_train, y_train)

    # predição
    y_pred = pipeline.predict(X_test)

    # métricas
    acc = accuracy_score(y_test, y_pred)

    resultados[nome] = acc

    print(f"\n📊 {nome}")
    print(f"Acurácia: {acc:.4f}")
    print(classification_report(y_test, y_pred))

In [ ]:
# ===============================
# 🏆 COMPARAÇÃO FINAL
# ===============================
print("\n🏆 Comparação de modelos:")
for nome, acc in resultados.items():
    print(f"{nome}: {acc:.4f}")

# Mostrando modelo de Decision Trees

In [ ]:
pipeline = Pipeline([
    ("preprocessamento", preprocessor),
    ("modelo", DecisionTreeClassifier(random_state=42, max_depth=3))
])

pipeline.fit(X_train, y_train)

arvore = pipeline.named_steps["modelo"]

feature_names = pipeline.named_steps["preprocessamento"].get_feature_names_out()

# mostrar a árvore
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(20,10))

plot_tree(
    arvore,
    feature_names=feature_names,
    class_names=["Não sobreviveu", "Sobreviveu"],
    filled=True,
    rounded=True
)

plt.title("Árvore de Decisão - Titanic")
plt.show()


# mostrar regras
def extrair_regras(arvore, feature_names):
    from sklearn.tree import _tree

    tree_ = arvore.tree_
    features = feature_names

    def recurse(node, regra):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = features[tree_.feature[node]]
            threshold = tree_.threshold[node]

            # esquerda
            recurse(tree_.children_left[node],
                    regra + [f"{name} <= {threshold:.2f}"])

            # direita
            recurse(tree_.children_right[node],
                    regra + [f"{name} > {threshold:.2f}"])
        else:
            classe = np.argmax(tree_.value[node])
            print("IF " + " AND ".join(regra) + f" THEN class = {classe}")

    recurse(0, [])

# executar
extrair_regras(arvore, feature_names)

# Visualizando os dados de acordo com a decision tree

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_plot = X.copy()
X_plot["Sex_num"] = X_plot["Sex"].map({"male": 0, "female": 1})

# jitter no eixo Y
jitter = np.random.normal(0, 0.05, size=len(X_plot))

plt.figure(figsize=(8,6))

# morreu
plt.scatter(
    X_plot["Age"][y == 0],
    X_plot["Sex_num"][y == 0] + jitter[y == 0],
    color="#ff9999",
    alpha=0.6,
    label="Died"
)

# sobreviveu
plt.scatter(
    X_plot["Age"][y == 1],
    X_plot["Sex_num"][y == 1] + jitter[y == 1],
    color="#99ff99",
    alpha=0.6,
    label="Survived"
)

plt.yticks([0,1], ["Male", "Female"])

plt.xlabel("Age")
plt.ylabel("Sex")
plt.title("Survival Patterns by Age and Sex (Titanic)")

plt.legend()
plt.grid(True)

plt.show()

# Random Forest

In [ ]:
pipeline = Pipeline([
    ("preprocessamento", preprocessor),
    ("modelo", RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42))
])

pipeline.fit(X_train, y_train)

arvore = pipeline.named_steps["modelo"]

feature_names = pipeline.named_steps["preprocessamento"].get_feature_names_out()

from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

for i in range(0,5):
  # pegar uma árvore da floresta
  arvore_rf = pipeline.named_steps["modelo"].estimators_[0]

  plt.figure(figsize=(20,10))

  plot_tree(
      arvore_rf,
      feature_names=feature_names,
      class_names=["Morreu", "Sobreviveu"],
      filled=True,
      rounded=True
  )

  plt.title(f"Árvore {i} da Random Forest")
  plt.show()